# Inside the Model: K-Means

## Same data, different starting centres

**30–35 minutes**

K-means alternates two simple decisions: assign each observation to a centre, then move each centre to its group's mean. Follow those decisions on six points and investigate whether changing only the start changes the result.

### Learning objectives

- assign observations to their nearest centre and update the two means;
- use the next assignment step to check whether a run has stopped;
- compare two final results using the lecture's inertia criterion.

Start after the lecture's K-means introduction. Commit to predictions before calculating. The optional advanced challenge adds about 10–15 minutes.


### Lecture connection and short recap

Revisit **Lecture 7 → “K-Means Clustering”** and **“Elbow Method”**. The lecture already gives the assignment/mean update cycle and defines inertia.

- **Assign:** send each observation to its nearest centre.
- **Update:** replace each centre by the mean of the observations assigned to it.
- **Repeat:** check the assignments again using the new centres.

The observations lie on one number line:

$$x=(0,\;1,\;2,\;5,\;6,\;10).$$

Use two clusters, equally weighted observations and ordinary distance on the line. For two ordered centres, their midpoint separates the nearest-centre regions. If a distance tie occurs, assign it to the left centre. No tie or empty cluster occurs in the two specified runs.

**Inertia** is the sum of squared distances to the assigned centres:

$$J=\sum_{i=1}^{6}(x_i-\text{centre assigned to }x_i)^2.$$

Smaller inertia means a closer fit to this criterion for the same data and number of clusters. Cluster numbers are just names, not known classes.

---


## 1. Predict the first move

**About 4 minutes · No Python**

Start the centres at **0 and 6**. Sketch the points and centres. Predict which points belong together and which way each centre will move when replaced by its group's mean.

Do you expect the isolated point at 10 to have its own cluster? Explain your prediction using the starting centres.

**Your prediction and sketch:** _Write here before calculating._


## 2. Assign, move, then check

**About 8 minutes · No Python**

Complete the first column of the table. Use the midpoint to make the assignment step quick. After moving the centres, calculate the **new** midpoint and check every point again.

| Step | Start: 0 and 6 | Second start: complete in Section 3 |
|:--|:--|:--|
| Initial midpoint | … | … |
| Points assigned to each centre | … | … |
| Updated centres: the two means | … | … |
| New midpoint | … | … |
| Does any point change group now? | … | … |
| Inertia using the updated centres | … | … |

For the last row, add the squared distances within each group and then add the two group totals. A calculator is fine. The main question is why the assignments do or do not change.

If the assignments stay the same, would another mean update move either centre? Explain without doing more arithmetic.

**Your trace and stopping explanation:** _Write here._


## 3. Change only the start

**About 8 minutes · No Python**

Reset the algorithm on the same six points with the same two clusters. This time start the centres at **6 and 10**.

Before calculating, predict whether this run will finish with the same groups. Explain why starting a centre at 10 might matter even though no data point has changed.

**Your second prediction:** _Write here before completing the table._

Complete the second column. Compare the two final inertias. For the run with the larger inertia, identify a better grouping already found in your own work.

Explain how a run can stop even though that better grouping exists. Use the nearest-centre decisions at the end of this run; do not assume that the algorithm tries every possible grouping.

**Your comparison:** _Refer to the points that would need to change group._


---

## 4. Verify the assignments and means

**About 8 minutes · Python now**

Complete three marked expressions in the same short loop for both starts. `labels` should be 0 for the left centre and 1 for the right centre. Comparing the two distance arrays gives a Boolean array; `.astype(int)` converts it to 0/1.

Then calculate the mean of `x[labels == 0]` and of `x[labels == 1]`. The remaining code checks the next assignment and compares your result with `KMeans`. Explicit initial centres and `n_init=1` keep the two starts separate.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

x = np.array([0, 1, 2, 5, 6, 10], dtype=float)
starts = [(0, 6), (6, 10)]


In [ ]:
final_centres = []
partitions = []
inertias = []

for start in starts:
    distance_left = np.abs(x - start[0])
    distance_right = np.abs(x - start[1])
    labels = None  # ADD YOUR CODE HERE: 1 when the right centre is closer.
    if labels is None:
        print('Complete the assignment expression, then run this cell again.')
        break
    mean_left = None  # ADD YOUR CODE HERE: mean of the points labelled 0.
    mean_right = None  # ADD YOUR CODE HERE: mean of the points labelled 1.
    if mean_left is None or mean_right is None:
        print('Complete the two means, then run this cell again.')
        break

    centres = np.array([mean_left, mean_right])
    midpoint = centres.mean()
    labels_after = (x > midpoint).astype(int)
    inertia = np.sum((x - centres[labels]) ** 2)
    final_centres.append(centres)
    partitions.append(labels)
    inertias.append(inertia)
    print('Start:', start, '| groups:', x[labels == 0], x[labels == 1])
    print('New centres:', centres, '| next assignment unchanged:',
          np.array_equal(labels, labels_after), '| inertia:', round(inertia, 3))
    model = KMeans(n_clusters=2, init=np.array(start, dtype=float).reshape(2, 1),
                   n_init=1, algorithm='lloyd', tol=1e-12, random_state=42)
    model.fit(x.reshape(-1, 1))
    print('Library centres:', np.sort(model.cluster_centers_.ravel()),
          '| inertia:', round(model.inertia_, 3))
    print()


In [ ]:
if len(final_centres) == 2:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True, sharey=True)
    for ax, start, centres, labels in zip(axes, starts, final_centres, partitions):
        for group, colour in zip([0, 1], ['tab:blue', 'tab:orange']):
            points = x[labels == group]
            ax.scatter(points, np.zeros(len(points)), color=colour, s=65)
        ax.scatter(centres, [0.5, 0.5], color='red', marker='X', s=130)
        ax.axvline(centres.mean(), color='grey', linestyle='--', alpha=0.7)
        ax.set_title('Final result from start ' + str(start))
        ax.set_xlabel('Observation value')
        ax.set_xlim(-0.8, 10.8)
        ax.set_ylim(-0.3, 0.9)
        ax.set_yticks([0, 0.5])
        ax.set_yticklabels(['Observations', 'Final centres'])
        ax.grid(axis='x', alpha=0.2)
    plt.tight_layout()
    plt.show()


The two vertical levels separate observations from centres; they are **not a second feature**. The dashed line is the final midpoint. Check why it causes no reassignment in either panel.

**Optional visual replay:** [Open after your calculation](../../Animations/Day7_KMeans_Replay.html) to replay assignment and centre movement. The notebook does not need this file to execute.


---

## 5. Explain the two endings

**About 5 minutes**

1. The data, distance and number of clusters were unchanged. Which decision in the first assignment step made the two runs different?
2. Why does “the centres stopped moving” not establish “we found the smallest possible inertia”? Point to evidence in your table.
3. If you tried several starts and kept the smallest inertia found, what would improve? What would that still not tell you about whether the groups are useful for a scientific question?

**Your interpretation:** _Use two or three sentences for each answer._

## Optional advanced challenge: establish the minimum

**Allow a further 10–15 minutes · No additional Python required**

Can you establish the smallest possible inertia for these six observations, rather than compare only two starts?

Use the midpoint rule to explain why an optimum can be chosen as **two nonempty consecutive blocks** of the sorted observations. Then consider the five cuts after 0, 1, 2, 5 and 6. For each cut, use the two block means and calculate the total inertia.

| Cut after | First block mean | Second block mean | Total inertia |
|--:|--:|--:|--:|
| 0 | … | … | … |
| 1 | … | … | … |
| 2 | … | … | … |
| 5 | … | … | … |
| 6 | … | … | … |

For shorter arithmetic, a block's squared-distance sum around its mean is

$$\sum_{i\in B}x_i^2-\frac{\left(\sum_{i\in B}x_i\right)^2}{|B|}.$$

Explain why checking these cuts covers the required alternatives. What additional claim does your completed comparison justify that the two stopped runs alone did not?

This optional argument extends the lecture's nearest-centre and mean-update ideas; the certificate is for this one-dimensional, two-cluster problem.

**Your argument and comparison:** _Write here._

---

## Takeaway

Complete: **“For K-means, the starting centres matter because …; stopping tells me …”** Distinguish a stopped run from a comparison of alternative results.
